# 06B — Geocoding integration and spatial checks

**Input:** `Results/05.xlsx`, `Results/06A_Final_Geocoding_Catalog.csv`
**Output:** `Results/06.xlsx`

Merges only the high-confidence coordinates back into the panel; matches found
outside the project's countries are held back rather than used. Point-to-point
Haversine distances are then computed to flag implausible line lengths for
manual review.

In [1]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

In [2]:
df = pd.read_excel("Results/05.xlsx", header=[0, 1], index_col=0)
catalog = pd.read_csv("Results/06A_Final_Geocoding_Catalog.csv", sep=';')

In [3]:
verified_mask = catalog['Status'].str.contains("Found in", na=False)
df_verified = catalog[verified_mask].copy()

uncertain_mask = (catalog['Status'] == "Found (Global Search)")
df_uncertain = catalog[uncertain_mask].copy()

missing_mask = catalog['Status'].isin(["Not Found", "Error"])
df_missing = catalog[missing_mask].copy()


### Add the coordinates to the initial database

In [4]:
verified_mask = catalog['Status'].str.contains("Found in", na=False)
verified_catalog = catalog[verified_mask]

lat_map = verified_catalog.set_index('Name')['Lat'].to_dict()
lon_map = verified_catalog.set_index('Name')['Lon'].to_dict()

for direction in ['From', 'To']:
    name_col = ('meta', f'Inv_Substation_{direction}_Final')
    
    lat_col = ('meta', f'Inv_{direction}_Lat')
    lon_col = ('meta', f'Inv_{direction}_Lon')
    
    df[lat_col] = df[name_col].map(lat_map)
    df[lon_col] = df[name_col].map(lon_map)

df = df.copy()

total_rows = len(df)
mapped_count = df[df[('meta', 'Inv_From_Lat')].notna()].shape[0]

print(f"Total projects in df: {total_rows}")
print(f"Projects with verified 'From' coordinates: {mapped_count} ({mapped_count/total_rows:.1%})")

Total projects in df: 965
Projects with verified 'From' coordinates: 622 (64.5%)


In [5]:
df.to_excel("Results/06.xlsx")

### Double check for long lines

In [6]:
def calculate_distance(row):
    lat1, lon1 = row[('meta', 'Inv_From_Lat')], row[('meta', 'Inv_From_Lon')]
    lat2, lon2 = row[('meta', 'Inv_To_Lat')], row[('meta', 'Inv_To_Lon')]
    
    if pd.isna(lat1) or pd.isna(lat2):
        return np.nan
    
    R = 6371.0 
    
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    
    return R * c

df[('meta', 'Line_Length_KM')] = df.apply(calculate_distance, axis=1)

suspicious_lines = df[df[('meta', 'Line_Length_KM')] > 500].copy()

In [7]:
medium_lines = df[
    (df[('meta', 'Line_Length_KM')] >= 400) & 
    (df[('meta', 'Line_Length_KM')] <= 500)
].copy()

medium_lines_sorted = medium_lines.sort_values(by=('meta', 'Line_Length_KM'), ascending=False)
